# 05_Monte_Carlo_bracket_simulations.ipynb

The objective of this notebook is to explore the use of Monte Carlo methods to simulate playoff brackets, given our Elo ratings.

# Desired output of simulations

For our complete system of choosing the optimal roster, the primary objective of the simulations is to output the expected number of series each team will make in the playoffs, given their starting bracket.

By consequence of this goal we will also be able to predict who the most probable Stanley Cup winner will be, which is fun.

Also of note is that we can through the simulations go a step further and estimate the total number of games each team will play. However, I have a feeling like this precise of an estimate may not be helpful because of randomness in real life and innaccuracies in modelling assumptions. We can just use our estimated number of series from simulations and then multiply by an assumption that every series will last six games (or go through playoff data and see exactly on average how long each series is).


# NHL playoffs format
The Stanley Cup playoffs consists of four rounds of best-of-seven series. Each series is aplyed in a 2-2-1-1-1 format, meaning the team with home-ice advantage hosts games one, two, five, and seven and the other team hosts games three, four, and six. Games five, six, and seven are only played if needed.

## Teams
There are 16 teams in total, with eight teams in each conference qualifying for the playoffs.

The first, second, and third placed teams in each of the four divisions qualify for the playoffs automatically. Two additional teams from each conference (regardless of division) make the playoffs as wildcards.

## First Round
Teams are split into two brackets per conference. Each bracket consists of the three divisional qualifiers and one of the wild cards.

The division winner with the best record plays the lowest seeded wildcard. The other division winner plays against the other wildcard. The other two series match the second and third place teams from their respective divisions.

So each conference will have a bracket set up like 

Division 1: 
- D1 vs. WC1 
- D2 vs. D3

Division 2: 
- D1 vs. WC2 
- D2 vs. D3

Where the D1, D2, D3 are their seeds in their respective divisions.

The higher seeded team in each matchup has home ice advantage.

### Seeding 
Seeding is relative to it's own divisional bracket. We have seeds 1-4 for each division:

Division 1:
- D1 (seed 1)
- D2 (seed 2)
- D3 (seed 3)
- WC  (seed 4)

## Second Round
The winners of the first round advance to the second round.

Let's say that in the Division 1 example above, WC1 and D2 win their respective series. These two team will play each other in the second round.

The higher seeded team in each matchup has home ice advantage.

## Third Round
The winners of the second round advance to the third round.

The third round are the conference finals.

The team with the better regular season record (regardless of seeding) has home ice advantage.

## Fourth Round
The winners of the third round advance to the fourth round.

The fourth round are the Stanley Cup Finals.

The team with the better regular season record (regardless of seeding) has home ice advantage.

## Tie-Breaking Procedure

In the case of ties in the regular season record, the league's [tie-breaking procedure](https://en.wikipedia.org/wiki/Season_structure_of_the_NHL#Stanley_Cup_playoffs) is applied. It is based on regulation wins first, and then further criteria if needed.

#### Notes
The team with "the better regular season record" is the team with more points.





# Monte Carlo Methods

At a high level, a Monte Carlo simulation is a computational algorithm that uses repeated random sampling to obtain the likelihood of a range of results of occuring. They are a way to model the probability of different outcomes of a process that can not be easily predicted.

## Simulating one game

Let's start with just trying to simulate one game.

We will have as input to a simulation:
- Home team Elo rating
- Away team Elo rating
- Knowledge of who is the home team (for the home advantage boost)

With this information, we can compute the probability the home team wins, $p_H$. We will then generate a random number (uniform distribution) $u$. If $u < p_H$, then the home team wins, otherwise the away team wins. If we run this many times and collect the results and compute probabilities. This gets more powerful when we try simulating entire series, brackets, playoffs.

### Read in data
First, let's get our Elo ratings for the 2024-2025 regular season.

In [1]:
from nhl_pool.config import PROCESSED_DIR
import json

ratings_path = PROCESSED_DIR / "20242025" / "regular" / "ratings.json"
with open(ratings_path, 'r') as file:
    ratings = json.load(file)
ratings["EDM"]

1552.4556745614689

The 2024-2025 Stanley Cup finals were between the Florida Panthers and the Edmonton Oilers, so let's start with these two teams.

In [2]:
def elo_compute_win_probability(R_h, R_a, h_adv=30):
    '''Given the Elo ratings for the home and away teams (h and a), compute the probability team h wins.'''
    return 1.0 / (1.0 + 10**((R_a - (R_h + h_adv))/400.0))

In [3]:
import numpy as np
rng = np.random.default_rng(seed=123)
    
def simulate_game(R_h, R_a, rng, h_adv=30):
    '''Returns a boolean 1 if home team won, otherwise home team lost.'''
    
    # Calculate the win probability
    p_home_win = elo_compute_win_probability(R_h, R_a, h_adv=h_adv)
    
    # Random number
    u = rng.random()
    
    return u < p_home_win

We'll simulate a game where EDM is the home team

In [4]:
R_h = ratings["EDM"]
R_a = ratings["FLA"]

p_home_win = elo_compute_win_probability(R_h, R_a)
print(f"Probability EDM win home game: {p_home_win:.4f}")

Probability EDM win home game: 0.5889


Now let's run this many times and compute the probability

In [5]:
n = 100000

wins_EDM = np.zeros(n)

for i in range(n):
    wins_EDM[i] = simulate_game(R_h, R_a, rng)

E_h = np.sum(wins_EDM) / n
E_a = 1 - E_h

print(f"Probability EDM win home game: {E_h:.4f}")
print(f"Probability FLA win away game: {E_a:.4f}")

Probability EDM win home game: 0.5893
Probability FLA win away game: 0.4107


As we might expect, when we run this simple simulation many times we almost recover our direct calculation of home team win probability, $p_H$. This is a good sanity check and can now move on to more complex simulations. 

# Simulating a series

With the confirmation that our simulation method works, we can try and simulte the entire series.

In the 2024-2025 Stanley Cup Finals, EDM was the team with home ice advantage for the best of seven 2-2-1-1-1 format.

Let's try and simulate the entire finals series.


In [6]:
def build_series_home_away_order(homeTeamAbbrev, awayTeamAbbrev):
    '''The homeTeamAbbrev is the teamAbbrev for the team with home ice advantage in the 2-2-1-1-1 format.'''
    # For each game type, the first team will be the home team for that game.
    g1 = (homeTeamAbbrev, awayTeamAbbrev)
    g2 = (awayTeamAbbrev, homeTeamAbbrev)
    
    return [g1, g1, g2, g2, g1, g2, g1] # 2-2-1-1-1


def init_series_outcome_dict(homeTeamAbbrev, awayTeamAbbrev):
    series_outcome = {homeTeamAbbrev: 0,
                   awayTeamAbbrev: 0}
    return series_outcome

def simulate_series(homeTeamAbbrev, awayTeamAbbrev, ratings, rng):
    game_order = build_series_home_away_order(homeTeamAbbrev, awayTeamAbbrev)
    
    series_outcome = init_series_outcome_dict(homeTeamAbbrev, awayTeamAbbrev)
    
    for game in game_order:
        gameHomeTeamAbbrev = game[0]
        gameAwayTeamAbbrev = game[1]
        
        # Extract ratings
        R_h = ratings[gameHomeTeamAbbrev]
        R_a = ratings[gameAwayTeamAbbrev]
        
        # Simulate_game
        home_win = simulate_game(R_h, R_a, rng)
        
        # Update series dict
        series_outcome[gameHomeTeamAbbrev] += home_win
        series_outcome[gameAwayTeamAbbrev] += 1 - home_win
        
        # Exit series simulation if either team has four wins
        if series_outcome[homeTeamAbbrev] == 4 or series_outcome[awayTeamAbbrev] == 4:
            return series_outcome

series_outcome = simulate_series("EDM", "FLA", ratings, rng)
print(series_outcome)

{'EDM': 4, 'FLA': 0}


Now let's simulate this series many times.

In [7]:
series_wins = {
    "EDM": 0,
    "FLA": 0
}

n = 200000

for _ in range(n):
    # Simulate series
    series_outcome = simulate_series("EDM", "FLA", ratings, rng)

    # Determine winner of series
    series_winner_abbrev = max(series_outcome, key=series_outcome.get)
    
    # Update aggregation
    series_wins[series_winner_abbrev] += 1

print(f"Probability EDM wins series: {series_wins["EDM"] / n :.4f}")
print(f"Probability FLA wins series: {series_wins["FLA"] / n:.4f}")

Probability EDM wins series: 0.6136
Probability FLA wins series: 0.3864


According to the simulations, there is a ~61% chance that EDM wins the Stanley Cup.

# Updating Elo during the simulation

Instead of holding the ratings static, for a single simulation run let's try updating the Elo ratings. This way we could try and capture team strength throughout the playoffs as well, for example sometimes teams go on hot streaks. For the next simulation, the ratings will reset to the team's base ratings.

## K-factor
In the Elo ratings calculations, we had a default of $K=20$ for the regular season ratings updates. For the playoffs and the stability of the simulations, the K-factor should probably be reduced such that ratings do not get big swings at this stage. Perhaps $K = 5$ or $K =10$ would be suitable. The playoff Elo updates are driven by simulated outcomes rather than observed results, so we reduce $K$ to avoid amplifying Monte Carlo induced-variance.

## Goal index, OT/SO multipliers
In the Elo ratings calculations we include a goal index and OT/SO multipliers in our ratings. However, in our simulations we only receive the game outcome. Therefore, we will simplify our rating update to omit these.

In [8]:

def elo_update_home_rating(R_h, E_h, S_h, K=20):
    return float(R_h + K*(S_h - E_h))

def elo_update_away_rating(R_a, E_a, S_a, K=20):
    return float(R_a + K*(S_a - E_a))

def elo_update_ratings(homeTeamAbbrev, awayTeamAbbrev, ratings, home_team_win, K=20, h_adv=30):
    
    # Get current ratings
    R_h = ratings[homeTeamAbbrev]
    R_a = ratings[awayTeamAbbrev]
    
    # Compute home win probability
    E_h = elo_compute_win_probability(R_h, R_a, h_adv=h_adv)
    E_a = 1 - E_h
    
    # Outcome
    S_h = float(int(home_team_win))   # 1.0 if home_win else 0.0
    S_a = 1.0 - S_h
    
    # Updates
    ratings[homeTeamAbbrev] = R_h + K * (S_h - E_h)
    ratings[awayTeamAbbrev] = R_a + K * (S_a - E_a)

    return ratings

In [9]:
def simulate_series(homeTeamAbbrev, awayTeamAbbrev, ratings, rng, K=10, h_adv=30):
    game_order = build_series_home_away_order(homeTeamAbbrev, awayTeamAbbrev)
    
    series_outcome = init_series_outcome_dict(homeTeamAbbrev, awayTeamAbbrev)
    
    for gameHomeTeamAbbrev, gameAwayTeamAbbrev in game_order:
        
        # Extract ratings
        R_h = ratings[gameHomeTeamAbbrev]
        R_a = ratings[gameAwayTeamAbbrev]
        
        # Simulate_game
        home_win = simulate_game(R_h, R_a, rng, h_adv=h_adv)
        
        # Update ratings
        ratings = elo_update_ratings(gameHomeTeamAbbrev, gameAwayTeamAbbrev, ratings, home_win, K=K, h_adv=h_adv)
        
        # Update series dict
        series_outcome[gameHomeTeamAbbrev] += home_win
        series_outcome[gameAwayTeamAbbrev] += 1 - home_win
        
        # Exit series simulation if either team has four wins
        if series_outcome[homeTeamAbbrev] == 4 or series_outcome[awayTeamAbbrev] == 4:
            return series_outcome


In [10]:
# Get base ratings
ratings_path = PROCESSED_DIR / "20242025" / "regular" / "ratings.json"
with open(ratings_path, 'r') as file:
    base_ratings = json.load(file)

# Init
series_wins = {
    "EDM": 0,
    "FLA": 0
}

# Simulate
n = 200000
for _ in range(n):
    # Reset ratings
    ratings = base_ratings.copy()
    
    # Simulate series
    series_outcome = simulate_series("EDM", "FLA", ratings, rng, K=10)

    # Determine winner of series
    series_winner_abbrev = max(series_outcome, key=series_outcome.get)
    
    # Update aggregation
    series_wins[series_winner_abbrev] += 1

print(f"Probability EDM wins series: {series_wins["EDM"] / n :.4f}")
print(f"Probability FLA wins series: {series_wins["FLA"] / n:.4f}")

Probability EDM wins series: 0.6036
Probability FLA wins series: 0.3964


With the inclusion of updating the Elo rating, we see that the probability of EDM winning the series dropped slightly.

# Simulating the entire playoffs.

With the framework for simulating one series complete, we will move on to simulating the entire playoffs.

The general approach of one simulation will be:
- Reset to base ratings
- For each round
    - Build the series/current matchups
    - Simulate each series best of seven
    - Update Elo after each game
- Record results (Stanley Cup winner, number of rounds reached for each team)

And then repeat that $N$ times.

## Seeding
I can circumvent some of the logic in the introduction of this notebook around seeding by giving the teams a seed 1 thru 16 and then hard-code the starting bracket. As the playoff sumlation progresses, the home/away order can be determined by choosing the team with the highest playoff overall seed.


In [11]:
def compute_playoff_seeds(standings, playoff_teams):
    # Filter the regular season standings to only include the playoff teams
    standings = standings[standings["teamAbbrev"].isin(playoff_teams)]

    # This is the order of which seeding tie-breakers are applied
    seeding_col_order = ["points", "regulationWins", "regulationAndOvertimeWins", "wins", "goalsDiff", "goalsFor"]

    standings['seed'] = (standings.sort_values(by=seeding_col_order, ascending=False)
                    .groupby(seeding_col_order, sort=False).ngroup().add(1))
    
    # Convert to dictionary
    seeds = standings[["teamAbbrev", "seed"]].set_index('teamAbbrev').to_dict()
    # unpack and return
    return seeds["seed"]
    

In [12]:
import pandas as pd
standings = pd.read_csv(PROCESSED_DIR / "20242025" / "regular" / "standings.csv")

playoff_teams = ["WPG", "STL", "DAL", "COL", "VGK", "MIN", "LAK", "EDM",
                "TOR", "OTT", "TBL", "FLA", "WSH", "MTL", "CAR", "NJD"]

seeds = compute_playoff_seeds(standings, playoff_teams)
seeds

{'WPG': 1,
 'WSH': 2,
 'VGK': 3,
 'TOR': 4,
 'DAL': 5,
 'LAK': 6,
 'TBL': 7,
 'COL': 8,
 'EDM': 9,
 'CAR': 10,
 'FLA': 11,
 'OTT': 12,
 'MIN': 13,
 'STL': 14,
 'MTL': 16,
 'NJD': 15}


## Brackets

We could code some logic to try and code the starting brackets (with additional conference data), but I think it will be easier to just hard-code the starting bracket.

The "from" keys indicate that the winner of that previous series will be in that matchup. Home and away team to be determined by seed.

In [13]:
bracket = {
    "R1":
        [
            # Western Conference R1
            {"id": "R1W1", "homeTeamAbbrev": "WPG", "awayTeamAbbrev": "STL"},
            {"id": "R1W2", "homeTeamAbbrev": "DAL", "awayTeamAbbrev": "COL"},
            {"id": "R1W3", "homeTeamAbbrev": "VGK", "awayTeamAbbrev": "MIN"},
            {"id": "R1W4", "homeTeamAbbrev": "LAK", "awayTeamAbbrev": "EDM"},
            
            # Eastern Converence R1
            {"id": "R1E1", "homeTeamAbbrev": "TOR", "awayTeamAbbrev": "OTT"},
            {"id": "R1E2", "homeTeamAbbrev": "TBL", "awayTeamAbbrev": "FLA"},
            {"id": "R1E3", "homeTeamAbbrev": "WSH", "awayTeamAbbrev": "MTL"},
            {"id": "R1E4", "homeTeamAbbrev": "CAR", "awayTeamAbbrev": "NJD"}
        ],
    "R2": 
        [
            # Western Conference R2
            {"id": "R2W1", "from": ("R1W1", "R1W2")},
            {"id": "R2W2", "from": ("R1W3", "R1W4")},
            
            # Eastern Conference R2
            {"id": "R2E1", "from": ("R1E1", "R1E2")},
            {"id": "R2E2", "from": ("R1E3", "R1E4")}
        ],
    "R3": 
        [
            # Conference Finals
            {"id": "CFW", "from": ("R2W1", "R2W2")},
            {"id": "CFE", "from": ("R2E1", "R2E2")}
        ],
    "R4":
        [
            # Stanley Cup Finals
            {"id": "SCF", "from": ("CFW", "CFE")}
        ]
}

# Simulating one playoff run

Let's do this rather verbose so we can make sure everything is running properly

In [14]:
def determine_team_order(seeds, teamA, teamB):
    if seeds[teamA] > seeds[teamB]:
        return teamA, teamB
    else:
        return teamB, teamA

In [15]:
# Intialize winners dict
winners = {}

# Read in standings and make copy
ratings_path = PROCESSED_DIR / "20242025" / "regular" / "ratings.json"
with open(ratings_path, 'r') as file:
    base_ratings = json.load(file)

ratings = base_ratings.copy()

# Simulate first round
for series in bracket["R1"]:
    # Simulate series
    series_outcome = simulate_series(series["homeTeamAbbrev"], series["awayTeamAbbrev"], ratings, rng)
    
    # Determine winner of series
    series_winner_abbrev = max(series_outcome, key=series_outcome.get)
    
    # Update winners dict
    winners[series["id"]] = series_winner_abbrev

# Simulate second round
for series in bracket["R2"]:    
    # Get the matchup from the previous winners
    teamA = winners[series["from"][0]]
    teamB = winners[series["from"][1]]
    
    # Determine who is the home team
    homeTeamAbbrev, awayTeamAbbrev = determine_team_order(seeds, teamA, teamB)
    
    # Simulate series
    series_outcome = simulate_series(homeTeamAbbrev, awayTeamAbbrev, ratings, rng)
    
    # Determine winner of series
    series_winner_abbrev = max(series_outcome, key=series_outcome.get)
    
    # Update winners dict
    winners[series["id"]] = series_winner_abbrev
    
# Simulate third round
for series in bracket["R3"]:    
    # Get the matchup from the previous winners
    teamA = winners[series["from"][0]]
    teamB = winners[series["from"][1]]
    
    # Determine who is the home team
    homeTeamAbbrev, awayTeamAbbrev = determine_team_order(seeds, teamA, teamB)
    
    # Simulate series
    series_outcome = simulate_series(homeTeamAbbrev, awayTeamAbbrev, ratings, rng)
    
    # Determine winner of series
    series_winner_abbrev = max(series_outcome, key=series_outcome.get)
    
    # Update winners dict
    winners[series["id"]] = series_winner_abbrev
    
# Simulate fourth round
for series in bracket["R4"]:    
    # Get the matchup from the previous winners
    teamA = winners[series["from"][0]]
    teamB = winners[series["from"][1]]
    
    # Determine who is the home team
    homeTeamAbbrev, awayTeamAbbrev = determine_team_order(seeds, teamA, teamB)
    
    # Simulate series
    series_outcome = simulate_series(homeTeamAbbrev, awayTeamAbbrev, ratings, rng)
    
    # Determine winner of series
    series_winner_abbrev = max(series_outcome, key=series_outcome.get)
    
    # Update winners dict
    winners[series["id"]] = series_winner_abbrev

In [16]:
winners


{'R1W1': 'WPG',
 'R1W2': 'COL',
 'R1W3': 'MIN',
 'R1W4': 'LAK',
 'R1E1': 'OTT',
 'R1E2': 'FLA',
 'R1E3': 'WSH',
 'R1E4': 'CAR',
 'R2W1': 'WPG',
 'R2W2': 'LAK',
 'R2E1': 'FLA',
 'R2E2': 'WSH',
 'CFW': 'WPG',
 'CFE': 'FLA',
 'SCF': 'FLA'}

Aggregating the number of rounds played:

Alternatively, could just make a counter go up in the simulation framework itself.

In [ ]:
rounds_played = {}
for team in playoff_teams:
    rounds_played[team] = 1

for series_id in winners:
    # Do not double count the SCF as an extra round
    if series_id != "SCF":
        rounds_played[winners[series_id]] += 1

rounds_played

{'WPG': 4,
 'STL': 1,
 'DAL': 1,
 'COL': 2,
 'VGK': 1,
 'MIN': 2,
 'LAK': 3,
 'EDM': 1,
 'TOR': 1,
 'OTT': 2,
 'TBL': 1,
 'FLA': 4,
 'WSH': 3,
 'MTL': 1,
 'CAR': 2,
 'NJD': 1}

# Simulating the playoffs many times
We have successfully simulated one run of the playoffs so it is now time to turn the above section into callable functions which we can then run many times. We will then aggregate the outcomes.



# Takeaways and Future Work

I think it would be interesting to add a per-game randomness factor. Hockey (along with any sport) is inherently difficult to model. Maybe if we added some variability on each game's simulation we could improve simulations. When we think of the delta used in the Elo win probability calculation, we could add some noise. Some pseudocode:
```
 delta = (R_h + h_adv) - R_a
if sigma_elo > 0:
    delta = delta + rng.normal(0.0, sigma_elo)

p_home_win = 1.0 / (1.0 + 10**(-delta / 400.0))
home_team_win = rng.random() < p_home_win
```

sigma_elo should be calibrated. Could do a grid search on noise amounts. Given ratings and regular season games (validation set), we could calculate log-loss of expected outcome and real outcomes for the different noise levels and try to minimise. 